In [1]:
#Bài 1(BTVN):
import copy
from heapq import heappush, heappop

# --- Cấu hình bài toán N-puzzle ---
# Đặt kích thước N cho bài toán N-puzzle (ví dụ: N=5 cho 24-puzzle, N=4 cho 15-puzzle)
kich_thuoc = 5 # Thay đổi giá trị này để thử với N khác (N > 1)

# Các hướng di chuyển: Dưới, Trái, Trên, Phải
huong_hang = [ 1, 0, -1, 0 ]
huong_cot = [ 0, -1, 0, 1 ]

class HangUuTien:
    def __init__(self):
        self.tap_tin = []
    def them(self, khoa):
        heappush(self.tap_tin, khoa)
    def lay(self):
        return heappop(self.tap_tin)
    def rong(self):
        return not self.tap_tin

class Nut:
    def __init__(self, cha, ma_tran, vi_tri_o_trong, chi_phi_h, cap_do):
        self.cha = cha
        self.ma_tran = ma_tran
        self.vi_tri_o_trong = vi_tri_o_trong
        self.chi_phi_h = chi_phi_h # h(n): số ô sai vị trí
        self.cap_do = cap_do # g(n): số bước đã đi
        # f(n) = g(n) + h(n)
        self.chi_phi_f = chi_phi_h + cap_do
    # So sánh để HangUuTien lấy nut có f(n) nhỏ nhất
    def __lt__(self, tiep_theo):
        return self.chi_phi_f < tiep_theo.chi_phi_f

def tinhH(ma_tran, dich) -> int:
    """Tính h(n): Số lượng ô nằm sai vị trí"""
    dem = 0
    for i in range(kich_thuoc):
        for j in range(kich_thuoc):
            if ma_tran[i][j] != 0 and ma_tran[i][j] != dich[i][j]:
                dem += 1
    return dem

def taoNutMoi(ma_tran, vi_tri_o_trong, vi_tri_o_trong_moi, cap_do, cha, dich) -> Nut:
    ma_tran_moi = copy.deepcopy(ma_tran)
    hang1, cot1 = vi_tri_o_trong
    hang2, cot2 = vi_tri_o_trong_moi
    # Hoán đổi ô trống với ô số
    ma_tran_moi[hang1][cot1], ma_tran_moi[hang2][cot2] = ma_tran_moi[hang2][cot2], ma_tran_moi[hang1][cot1]
    chi_phi_h = tinhH(ma_tran_moi, dich)
    return Nut(cha, ma_tran_moi, vi_tri_o_trong_moi, chi_phi_h, cap_do)

def inMaTran(ma_tran):
    for dong in ma_tran:
        print(" ".join(f"{gia_tri:2d}" for gia_tri in dong))

def kiemTraHopLe(hang, cot):
    return 0 <= hang < kich_thuoc and 0 <= cot < kich_thuoc

def inDuongDi(goc):
    if goc is None: return
    inDuongDi(goc.cha)
    inMaTran(goc.ma_tran)
    print(f"f(n)={goc.chi_phi_f} (g={goc.cap_do}, h={goc.chi_phi_h})\n")

def giaiBaiToan(trang_thai_dau, vi_tri_o_trong_ban_dau, trang_thai_dich):
    hang_uu_tien = HangUuTien()
    chi_phi_h_goc = tinhH(trang_thai_dau, trang_thai_dich)
    goc = Nut(None, trang_thai_dau, vi_tri_o_trong_ban_dau, chi_phi_h_goc, 0)
    hang_uu_tien.them(goc)
    da_tham = set() # Tránh lặp lại trạng thái đã xét

    # Để tránh lỗi nếu giải pháp quá dài hoặc bộ nhớ lớn
    # Có thể giới hạn số lượng nút duyệt hoặc thêm điều kiện dừng khác

    while not hang_uu_tien.rong():
        nut_min = hang_uu_tien.lay()
        # Nếu h(n) = 0 thì đã đạt trạng thái đích
        if nut_min.chi_phi_h == 0:
            inDuongDi(nut_min)
            return
        # Lưu trạng thái vào da_tham (chuyển matrix thành tuple để hash)
        # Sử dụng tuple để các ma trận có thể hash được trong set
        bo_ba_trang_thai = tuple(tuple(dong) for dong in nut_min.ma_tran)
        if bo_ba_trang_thai in da_tham:
            continue
        da_tham.add(bo_ba_trang_thai)
        for i in range(4):
            vi_tri_moi = [
                nut_min.vi_tri_o_trong[0] + huong_hang[i],
                nut_min.vi_tri_o_trong[1] + huong_cot[i]
            ]
            if kiemTraHopLe(vi_tri_moi[0], vi_tri_moi[1]):
                con = taoNutMoi(nut_min.ma_tran, nut_min.vi_tri_o_trong,
                               vi_tri_moi, nut_min.cap_do + 1, nut_min, trang_thai_dich)
                hang_uu_tien.them(con)
    print("Không tìm thấy đường đi (có thể do quá phức tạp hoặc không có giải pháp).")

# --- KHỞI TẠO BÀI TOÁN N-PUZZLE ---
# Tạo trạng thái đích và trạng thái ban đầu ví dụ tự động cho N-puzzle
def tao_trang_thai_dich(n):
    dich = []
    current_num = 1
    for i in range(n):
        row = []
        for j in range(n):
            if i == n - 1 and j == n - 1:
                row.append(0) # ô trống ở cuối cùng
            else:
                row.append(current_num)
                current_num += 1
        dich.append(row)
    return dich

def tao_trang_thai_khoi_tao_don_gian(n, dich_state):
    # Tạo một trạng thái ban đầu đơn giản bằng cách di chuyển ô trống một vài bước từ trạng thái đích
    khoi_tao = copy.deepcopy(dich_state)
    empty_r, empty_c = n - 1, n - 1 # Vị trí ô trống trong trạng thái đích

    # Ví dụ: Hoán đổi ô trống với ô bên trái nó (nếu có)
    if empty_c > 0:
        khoi_tao[empty_r][empty_c], khoi_tao[empty_r][empty_c - 1] = khoi_tao[empty_r][empty_c - 1], khoi_tao[empty_r][empty_c]
        vi_tri_o_trong_ban_dau = [empty_r, empty_c - 1]
    else: # Nếu ô trống ở cột 0, hoán đổi với ô bên trên nó
        khoi_tao[empty_r][empty_c], khoi_tao[empty_r - 1][empty_c] = khoi_tao[empty_r - 1][empty_c], khoi_tao[empty_r][empty_c]
        vi_tri_o_trong_ban_dau = [empty_r - 1, empty_c]

    return khoi_tao, vi_tri_o_trong_ban_dau

# --- Chạy thuật toán ---
trang_thai_dich = tao_trang_thai_dich(kich_thuoc)
trang_thai_khoi_tao, vi_tri_o_trong_ban_dau = tao_trang_thai_khoi_tao_don_gian(kich_thuoc, trang_thai_dich)

print("Trạng thái ban đầu:")
inMaTran(trang_thai_khoi_tao)
print("\nTrạng thái đích:")
inMaTran(trang_thai_dich)
print("\nBắt đầu giải bài toán N-puzzle với A*:\n")

giaiBaiToan(trang_thai_khoi_tao, vi_tri_o_trong_ban_dau, trang_thai_dich)


Trạng thái ban đầu:
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23  0 24

Trạng thái đích:
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23 24  0

Bắt đầu giải bài toán N-puzzle với A*:

 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23  0 24
f(n)=1 (g=0, h=1)

 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23 24  0
f(n)=1 (g=1, h=0)



In [2]:
#Bài 2(BTVN):
class GiaoHang:
  def __init__(self, danh_sach_dia_diem):
    self.danh_sach_dia_diem=danh_sach_dia_diem
  def lay_diem_gan_nhau(self, a):
    return self.danh_sach_dia_diem.get(a,[]) # Fixed: .get is a method, not item access
  def h(self, n): # Fixed: Renamed method from 'b' to 'h' for consistency
    h_score = {
          'A': 12,
          'B': 93,
          'C': 7,
          'D': 5
    }
    return h_score.get(n, 1) # Fixed: Used h_score instead of undefined b_score
  def thuat_toan(self, bat_dau, ket_thuc):
      open_list = set([bat_dau])
      closed_list = set([])
      g = {bat_dau: 0}
      cha = {bat_dau: bat_dau}
      while len(open_list) > 0:
        n = None
        for v in open_list:
          # Fixed: Changed 'a' to 'v' for correct variable scope
          if n == None or g[v] + self.h(v) < (g[n] + self.h(n) if n else float('inf')):
                    n = v
        if n== None:
            print("Không tìm thấy đường đi!")
            return None
        if n == ket_thuc:
              duong_di = []
              while cha[n] != n:
                  duong_di.append(n)
                  n = cha[n]
              duong_di.append(bat_dau)
              duong_di.reverse()
              print(f"Tìm thấy đường đi: {duong_di}")
              return duong_di
        for (m, trong_so) in self.lay_diem_gan_nhau(n):
          if m not in open_list and m not in closed_list:
            open_list.add(m)
            cha[m] = n
            g[m] = g[n] + trong_so
          else:
                  if g[m] > g[n] + trong_so:
                      g[m] = g[n] + trong_so
                      cha[m] = n
                      if m in closed_list:
                          closed_list.remove(m)
                          open_list.add(m)
        open_list.remove(n)
        closed_list.add(n)
      print("Đường đi không tồn tại!")
      return None

du_lieu_ke = {
    'A': [('B', 1), ('C', 3)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)]
}

do_thi = GiaoHang(du_lieu_ke)
do_thi.thuat_toan('A', 'D')

Tìm thấy đường đi: ['A', 'C', 'D']


['A', 'C', 'D']